In [1]:
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import optax
import xarray as xr
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
    DISCO,
    DPOT,
)
from context_flux_no.training import PDEDataset
from context_flux_no.training.loader import SegmentLoader
from jaxtyping import Array, Float, PRNGKeyArray

In [2]:
# jax.config.update("jax_enable_x64", True)

datadir = Path("../../data")
# dataset_xr = xr.load_dataset(
#     datadir / "cubic_no_source_large_train.hdf5", engine="h5netcdf"
# )
dataset_xr = xr.open_mfdataset(
    sorted(list(datadir.glob("cubic_no_source_large_train_seed=*.hdf5"))),
    combine="nested",
    concat_dim="pde",
    engine="h5netcdf",
).isel(t=slice(None, None, 10))
dataset_xr

<xarray.Dataset> Size: 8GB
Dimensions:  (pde: 1000, ic: 100, t: 101, dim: 1, x: 100, param: 3)
Coordinates:
  * t        (t) float64 808B 0.0 0.005 0.01 0.015 0.02 ... 0.485 0.49 0.495 0.5
  * x        (x) float64 800B 0.005 0.015 0.025 0.035 ... 0.975 0.985 0.995
  * dim      (dim) <U1 4B 'u'
  * param    (param) <U1 12B 'a' 'b' 'c'
Dimensions without coordinates: pde, ic
Data variables:
    values   (pde, ic, t, dim, x) float64 8GB dask.array<chunksize=(100, 100, 101, 1, 100), meta=np.ndarray>
    coeffs   (pde, param) float64 24kB dask.array<chunksize=(100, 3), meta=np.ndarray>

In [3]:
dataset_train, dataset_test = PDEDataset.from_xarray(dataset_xr).split_by_time(80)

/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/training/dataset.py:33: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  return cls(u, t, x, coeffs, dim_names, coeff_names)
/home/jhko725/.local/share/uv/python/cpython-3.12.9-linux-x86_64-gnu/lib/python3.12/dataclasses.py:1588: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  return obj.__class__(**changes)


In [4]:
dataset_train

PDEDataset(
  u=f64[1000,100,80,1,100](numpy),
  t=f64[80](numpy),
  x=f64[100](numpy),
  coeffs=f64[1000,3](numpy),
  dim_names=[np.str_('u')],
  coeff_names=[np.str_('a'), np.str_('b'), np.str_('c')]
)

In [5]:
loader = SegmentLoader(dataset_train, segment_length=21, batch_size=1024)
batch, _ = loader.load_batch(loader.init())

In [ ]:
batch[0].shape

(1024, 21, 1, 100)

In [ ]:
16000000

In [ ]:
dpot = DPOT(
    num_spatial_dims=1,
    in_channels=1,
    out_channels=1,
    in_timesteps=20,
    patch_size=(4,),
    img_size=(100,),
    embedding_dim=256,
    max_frequency_modes=(10,),
    fno_depth=5,
    num_blocks=4,
    num_classes=12,
    hidden_dim_patch=256,
    hidden_dim_fno=256,
    hidden_dim_output=32,
    key=jax.random.key(0),
)
dpot.num_parameters()  # 0.238M

2383469

In [6]:
disco = DISCO(
    num_spatial_dims=1,
    channels=1,
    embedding_dim=48,
    patch_size=100,
    num_hypernet_blocks=2,
    droppath=0.1,
    num_hypernet_heads=2,
    mlp_hidden_dim=48,
    boundary_condition="periodic",
    key=jax.random.key(0),
)
disco.num_parameters()  # 11.67M

8067638

In [7]:
def loss_fn(
    model: AbstractMultiphysicsOperator,
    batch: tuple[Float[Array, "..."], ...],
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u, dt, dx = batch
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred = eqx.filter_vmap(lambda u_, key_: model(u_, key=key_))(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


In [8]:
eqx.filter_value_and_grad(loss_fn, has_aux=True)(disco, batch, jax.random.key(0))

E0227 02:23:30.453396 2252694 hlo_lexer.cc:444] Failed to parse int literal: 368784927353064689048
E0227 02:23:30.453498 2252694 hlo_lexer.cc:444] Failed to parse int literal: 368784927353064689048
2026-02-27 02:23:40.982576: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-02-27 02:23:40.986870: E external/xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng28{k2=4,k3=0} for conv %cudnn-conv.13 = (f32[1,65536,12]{2,1,0}, u8[0]{0}) custom-call(%bitcast.75, %bitcast.76), window={size=3}, dim_labels=bf0_oi0->bf0, feature_group_count=65536, custom_call_target="__cudnn$convForward", metadata={op_name="jit(diffeqsolve)/jit(main)/outer-loop/checkpointed-fwd/while/body/checkpointed-no-vjp/while/body/eqx.nn.Conv/conv_general_dilated" source_file="/home/jhko725/projects/CONTEXT_FLUX_NO/.venv/lib/python3.12/site-packages/equinox/

((Array(0.00853133, dtype=float32), {}),
 DISCO(
   hypernet=HyperNetwork(
     adapter=Linear(
       weight=f32[12,1],
       bias=f32[12],
       in_features=1,
       out_features=12,
       use_bias=True
     ),
     encoders={
       1:
       Sequential(
         layers=(
           Conv(
             num_spatial_dims=1,
             weight=f32[12,12,25],
             bias=f32[12,1],
             in_channels=12,
             out_channels=12,
             kernel_size=(25,),
             stride=(25,),
             padding=((0, 0),),
             dilation=(1,),
             groups=1,
             use_bias=True,
             padding_mode='REFLECT'
           ),
           GroupNorm(
             groups=1,
             channels=12,
             eps=1e-05,
             channelwise_affine=True,
             weight=f32[12],
             bias=f32[12]
           ),
           Lambda(fn=None),
           Conv(
             num_spatial_dims=1,
             weight=f32[12,12,2],
             

In [9]:
def train(
    model: AbstractMultiphysicsOperator,
    dataloader: SegmentLoader,
    optimizer: optax.GradientTransformation,
    loss_fn,
    max_steps: int,
    *,
    key: PRNGKeyArray = jax.random.key(0),
):
    loss_grad_fn = eqx.filter_value_and_grad(loss_fn, has_aux=True)

    @eqx.filter_jit
    def train_step(model_, loader_state_, opt_state_, key_):
        batch, loader_state_next = dataloader.load_batch(loader_state_)
        key_, key_next = jax.random.split(key_)

        (loss, aux), grads = loss_grad_fn(model_, batch, key_)
        updates, opt_state_next = optimizer.update(
            grads, opt_state_, eqx.filter(model_, eqx.is_array)
        )
        model_ = eqx.apply_updates(model_, updates)
        return loss, aux, model_, loader_state_next, opt_state_next, key_next

    loader_state = dataloader.init()
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    loss_history = []
    # try:
    for i in range(max_steps):
        loss, aux, model, loader_state, opt_state, key = train_step(
            model, loader_state, opt_state, key
        )

        loss_scalar = loss.item()
        print(f"Step: {i}: loss = {loss_scalar}")
        loss_history.append(loss_scalar)
    # finally:
    return model, jnp.asarray(loss_history)

In [10]:
optimizer = optax.adamw(1e-3)
model, loss_history = train(disco, loader, optimizer, loss_fn, max_steps=5000)

/home/jhko725/projects/CONTEXT_FLUX_NO/.venv/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1132: UserWarning: A large amount of constants were captured during lowering (6.40GB total). If this is intentional, disable this warning by setting JAX_CAPTURED_CONSTANTS_WARN_BYTES=-1. To obtain a report of where these constants were encountered, set JAX_CAPTURED_CONSTANTS_REPORT_FRAMES=-1.
  warnings.warn(message)


Step: 0: loss = 0.008531332015991211
Step: 1: loss = 0.004759001079946756
Step: 2: loss = 0.0036931114736944437
Step: 3: loss = 0.0038416816387325525
Step: 4: loss = 0.0032882625237107277
Step: 5: loss = 0.0032355438452214003
Step: 6: loss = 0.002897353610023856
Step: 7: loss = 0.003101162612438202
Step: 8: loss = 0.0030137854628264904
Step: 9: loss = 0.0031334662344306707
Step: 10: loss = 0.0029978197999298573
Step: 11: loss = 0.0030484527815133333
Step: 12: loss = 0.0030047446489334106
Step: 13: loss = 0.003250346053391695
Step: 14: loss = 0.0028301512356847525
Step: 15: loss = 0.0030155866406857967
Step: 16: loss = 0.003098834306001663
Step: 17: loss = 0.0029587375465780497
Step: 18: loss = 0.002889947732910514
Step: 19: loss = 0.0028574352618306875
Step: 20: loss = 0.002894167322665453
Step: 21: loss = 0.00283824373036623
Step: 22: loss = 0.0029372081626206636
Step: 23: loss = 0.002732117660343647
Step: 24: loss = 0.0028874825220555067
Step: 25: loss = 0.0029547461308538914
Step: 2

KeyboardInterrupt: 